# sanitise - OCR metrics per domain, unattended (Kaggle "Save & Run All")

Runs the whole OCR evaluation by itself: one report per domain (health, finance, law) plus real
handwriting (IAM lines), then a summary table and a zip. Nothing here needs you at the screen.

**How to run it (laptop can be off afterwards):**
1. Right sidebar -> Session options: Accelerator **GPU T4 x2**, Internet **On**.
2. Change the settings in cell 1 if you want (defaults: every document, 100 IAM lines).
3. Top right -> **Save Version** -> **Save & Run All (Commit)** -> **Save**.
4. Close the tab. Kaggle runs it in the background (about 2-4 hours, uses GPU quota).
5. Later: open this notebook on Kaggle -> the new version -> **Output** tab -> download
   `ocr_results.zip` (or read `ocr_results/SUMMARY.md` there).

Each domain runs on its own, so if one fails the others still finish; `ocr_results/log.txt` says
what happened. Do not run it interactively and then also Save & Run All: each one uses GPU time.

In [ ]:
# 1 Settings and setup
BRANCH = "ingest"
OCR_MODEL = "qwen3-vl:8b-instruct"
DOMAINS = ["health", "finance", "law"]   # one report each
SETS = "data/ocr_test,data/ocr_real"    # synthetic + real; "data/ocr_real" for real only
N_PER_DOMAIN = None                      # None = every document; a number = that many per domain
REAL_PER_DOMAIN = 50                     # real documents to download per domain
IAM_LINES = 100                          # real handwritten lines (IAM); 0 = skip

import os
import subprocess
import sys
import time
from pathlib import Path

RESULTS = Path("/kaggle/working/ocr_results")
RESULTS.mkdir(parents=True, exist_ok=True)
LOG = RESULTS / "log.txt"


def log(msg):
    line = f"{time.strftime('%H:%M:%S')}  {msg}"
    print(line, flush=True)
    with open(LOG, "a") as f:
        f.write(line + "\n")


log(f"start: branch {BRANCH}, model {OCR_MODEL}, domains {DOMAINS}, sets {SETS}")
# the code and the models live outside /kaggle/working, so the saved output is only the results
os.environ["SANITISE_WORKDIR"] = "/tmp/sanitise"
REPO = Path("/tmp/3rd_sem_el")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/raksha-sendhil/3rd_sem_el.git", str(REPO)], check=True)
os.chdir(REPO)
commit = subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip()
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU"
log(f"code {commit} | GPU {gpu}")
if gpu == "NO GPU":
    log("WARNING: no GPU - set Accelerator to GPU T4 x2 in Session options; scanned pages will be very slow")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
sys.path.insert(0, str(REPO / "tools"))
import colab_cache

colab_cache.install_ollama_and_model(OCR_MODEL)
os.environ.update({"SANITISE_OCR_MODEL": OCR_MODEL, "SANITISE_LLM_BACKEND": "ollama"})
log("setup done")

In [ ]:
# 2 Download the real test set (50 per domain by default)
r = subprocess.run([sys.executable, "tools/fetch_real_testset.py", "--per-domain", str(REAL_PER_DOMAIN)],
                   capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
log("real test set: " + (r.stdout.strip().splitlines() or ["?"])[-1] if r.returncode == 0
    else f"real test set FAILED (code {r.returncode}); continuing with what exists")

In [ ]:
# 3 One evaluation per domain, then real handwriting (IAM); a failure in one does not stop the rest
def evaluate(name, extra):
    out = RESULTS / name
    cmd = [sys.executable, "tools/ocr_eval.py", "--set", SETS, "--out", str(out), "--quiet", *extra]
    log(f"{name}: started ({' '.join(extra)})")
    started = time.time()
    with open(RESULTS / f"{name}_progress.txt", "w") as progress:
        code = subprocess.run(cmd, stdout=progress, stderr=subprocess.STDOUT).returncode
    minutes = (time.time() - started) / 60
    log(f"{name}: {'done' if code == 0 else f'FAILED (code {code}), see {name}_progress.txt'} in {minutes:.0f} min")


for domain in DOMAINS:
    extra = ["--domain", domain, "--iam", "0"]
    if N_PER_DOMAIN is not None:
        extra += ["--n", str(N_PER_DOMAIN)]
    evaluate(domain, extra)
if IAM_LINES:
    evaluate("iam_handwriting", ["--n", "0", "--iam", str(IAM_LINES)])

In [ ]:
# 4 Summary table across domains, and a zip of everything
import json
import shutil

from IPython.display import Markdown, display


def pct(x):
    return "-" if x is None else f"{100 * x:.1f}%"


lines = ["# OCR metrics per domain", "", f"Model `{OCR_MODEL}`, code {commit}, test sets {SETS}.", "",
         "| Domain | Docs | Pages | CER | Word F1 | WER | Private details read exactly | Pages routed correctly | s / vision page |",
         "|---|---|---|---|---|---|---|---|---|"]
sources = ["", "## By source", "", "| Domain | Source | Docs | CER | Word F1 |", "|---|---|---|---|---|"]
for domain in DOMAINS:
    f = RESULTS / domain / "report.json"
    if not f.exists():
        lines.append(f"| {domain} | failed - see log.txt | | | | | | | |")
        continue
    s = json.loads(f.read_text())["summary"]
    a = s["all"]
    sec = f"{a['seconds_per_vision_page']:.1f}" if a.get("seconds_per_vision_page") else "-"
    lines.append(f"| {domain} | {a['docs']} | {a['pages']} | **{pct(a['cer'])}** | {pct(a['word_f1'])} | "
                 f"{pct(a['wer'])} | {pct(a['detail_recall'])} | {pct(a['route_accuracy'])} | {sec} |")
    for src, ss in s["sources"].items():
        sources.append(f"| {domain} | {src} | {ss['docs']} | {pct(ss['cer'])} | {pct(ss['word_f1'])} |")
f = RESULTS / "iam_handwriting" / "report.json"
if f.exists():
    iam = json.loads(f.read_text())["summary"].get("iam")
    if iam:
        lines += ["", f"**Real handwriting (IAM)**: {iam['lines']} lines, CER {pct(iam['cer'])}, "
                      f"WER {pct(iam['wer'])}."]
lines += sources
lines += ["", "Quote word F1 for SROIE, CORD and FUNSD (approximate gold order). Full tables per domain: "
              "`<domain>/report.md`."]
(RESULTS / "SUMMARY.md").write_text("\n".join(lines))
display(Markdown("\n".join(lines)))

# the readings per document stay in the zip too (pred/), for checking later
shutil.make_archive("/kaggle/working/ocr_results", "zip", RESULTS)
log(f"summary written; zip {Path('/kaggle/working/ocr_results.zip').stat().st_size / 1e6:.1f} MB - all done")